In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


# Pre - Processing

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/shwetabh123/mall-customers/Mall_Customers.csv')

df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-MEANS CLUSTERING
#### with 2 column

In [ ]:
# Find Optimal K (Elbow Method)
wcss = []

for i in range(1, 10):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 10), wcss)
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
y_kmeans = kmeans.fit_predict(X_scaled)

In [ ]:
print("KMeans Score:", silhouette_score(X_scaled, y_kmeans))

In [ ]:
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_kmeans, cmap='viridis')
plt.title("K-Means Clustering")
plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.show()

# DBSCAN CLUSTERING
#### with 2 column

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
y_dbscan = dbscan.fit_predict(X_scaled)

In [ ]:
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_dbscan, cmap='plasma')
plt.title("DBSCAN Clustering")
plt.xlabel("Income")
plt.ylabel("Spending Score")
plt.show()

In [ ]:

print("DBSCAN Score:", silhouette_score(X_scaled, y_dbscan))

# PCA

In [ ]:
# Drop CustomerID (not useful for clustering)
df = df.drop(['CustomerID'], axis=1)

# Convert Gender to numeric
df['Genre'] = df['Genre'].map({'Male': 0, 'Female': 1})

# Final feature set
X = df
X

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)

In [ ]:
print("Total Variance Retained:", sum(pca.explained_variance_ratio_))

# K-MEANS with 2 PCA component

In [ ]:
# K-MEANS ON PCA DATA
kmeans = KMeans(n_clusters=5, random_state=42)
y_kmeans = kmeans.fit_predict(X_pca)
plt.figure(figsize=(8,6))

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, cmap='viridis', s=50)

# Plot centroids
plt.scatter(kmeans.cluster_centers_[:, 0],
            kmeans.cluster_centers_[:, 1],
            color='red', s=200, marker='X', label='Centroids')

plt.title("K-Means Clustering (PCA Reduced Data)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()

plt.show()

# DBSCAN with 2 PCA component

In [ ]:
# DBSCAN ON PCA DATA
dbscan = DBSCAN(eps=0.5, min_samples=5)
y_dbscan = dbscan.fit_predict(X_pca)

plt.figure(figsize=(8,6))

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_dbscan, cmap='plasma', s=50)

plt.title("DBSCAN Clustering (PCA Reduced Data)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.show()

In [ ]:
print("KMeans Score:", silhouette_score(X_scaled, y_kmeans))
print("DBSCAN Score:", silhouette_score(X_scaled, y_dbscan))

# Apply PCA (3 Components)

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Total Variance:", sum(pca.explained_variance_ratio_))

# K-MEANS with 3 PCA component

In [ ]:
# K-MEANS
kmeans = KMeans(n_clusters=5, random_state=42)
y_kmeans = kmeans.fit_predict(X_pca)

fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
           c=y_kmeans, cmap='viridis', s=50)

# Plot centroids
centers = kmeans.cluster_centers_
ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2],
           c='red', s=200, marker='X', label='Centroids')

ax.set_title("3D K-Means Clustering (PCA)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")

ax.legend()
plt.show()

# DBSCAN with 3 PCA component

In [ ]:
dbscan = DBSCAN(eps=0.6, min_samples=5)
y_dbscan = dbscan.fit_predict(X_pca)

fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
           c=y_dbscan, cmap='plasma', s=50)

ax.set_title("3D DBSCAN Clustering (PCA)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")

plt.show()

In [ ]:
print("KMeans Score:", silhouette_score(X_scaled, y_kmeans))
print("DBSCAN Score:", silhouette_score(X_scaled, y_dbscan))